# From C# models to retrievable context

Compile annotated POCOs, apply fluent configuration, inspect relationships and ACL mappings, compare changes,
and deserialize a live authorized read into a C# object.

The first sections run offline after building the assemblies. Only the final read requires
[shared setup](00-context-retriever-setup.ipynb); no OpenAI key is needed. Start a fresh C# kernel in `notebooks/csharp`.
The small types below are **read projections** of the full travel fixture. Do not deploy this reduced model over
the shared surface: it omits fields and entities. Setup deploys the complete model from `TravelContext.cs`.


In [ ]:
#nullable enable
#r "nuget: Microsoft.Agents.AI, 1.23.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.10.1"
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.12"
#r "nuget: ModelContextProtocol, 2.2.0"
#r "../../src/Redis.Iris.AgentMemory/bin/Release/net8.0/Redis.Iris.AgentMemory.dll"
#r "../../src/Redis.Iris.ContextRetriever.Modeling/bin/Release/net8.0/Redis.Iris.ContextRetriever.Modeling.dll"
#r "../../src/Redis.Iris.ContextRetriever/bin/Release/net8.0/Redis.Iris.ContextRetriever.dll"
#r "../../src/Redis.Iris.AgentFramework/bin/Release/net8.0/Redis.Iris.AgentFramework.dll"

using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Text.Json;
using System.Text.Json.Serialization;
using System.Threading;
using System.Threading.Tasks;
using Microsoft.Agents.AI;
using Redis.Iris.AgentFramework;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.DependencyInjection;
using Redis.Iris.AgentMemory;
using Redis.Iris.ContextRetriever;
using Redis.Iris.ContextRetriever.Modeling;
using OpenAI;


In [ ]:
#load "context-notebook.csx"


## Declare a traveler and its related flight

`ContextField` opts a property into the model; `JsonPropertyName` separately controls runtime JSON deserialization.
`AccessTag` creates an ACL mapping, beyond merely adding a TAG index. The unannotated `Notes` property is excluded.
Model compilation reads type metadata and does not instantiate or persist entities.


In [ ]:
[ContextEntity(Name = "Traveler", KeyTemplate = "traveler:{id}", Description = "Travel customer")]
public sealed class TravelerView
{
    [ContextField(Name = "id", IsKeyComponent = true)]
    [JsonPropertyName("id")]
    public string Id { get; init; } = "";
    [ContextField(Name = "tenant_id", Index = ContextIndex.Tag, AccessTag = "tenant_id")]
    [JsonPropertyName("tenant_id")]
    public string TenantId { get; init; } = "";
    [ContextField(Name = "traveler_id", Index = ContextIndex.Tag, AccessTag = "traveler_id")]
    [JsonPropertyName("traveler_id")]
    public string TravelerId { get; init; } = "";
    [ContextField(Name = "full_name", Index = ContextIndex.Text)]
    [JsonPropertyName("full_name")]
    public string FullName { get; init; } = "";
    // Unannotated: excluded until explicitly added by fluent configuration.
    public string? Notes { get; init; }
}

[ContextEntity(Name = "FlightReservation", KeyTemplate = "flight-reservation:{id}")]
public sealed class FlightView
{
    [ContextField(Name = "id", IsKeyComponent = true)]
    public string Id { get; init; } = "";
    [ContextField(Name = "tenant_id", Index = ContextIndex.Tag, AccessTag = "tenant_id")]
    [JsonPropertyName("tenant_id")]
    public string TenantId { get; init; } = "";
    [ContextField(Name = "traveler_id", Index = ContextIndex.Tag, AccessTag = "traveler_id")]
    [JsonPropertyName("traveler_id")]
    public string TravelerId { get; init; } = "";
    [ContextRelationship(typeof(TravelerView), Name = "traveler", SourceField = "traveler_id")]
    [JsonIgnore]
    public TravelerView? Traveler { get; init; }
}


In [ ]:
var annotatedModel = ContextModelCompiler.Compile("Notebook travel projection", "Traveler and flight read shapes",
    typeof(TravelerView), typeof(FlightView));
Console.WriteLine(ContextDataModelJson.Serialize(annotatedModel));
if (annotatedModel.Entities.Single(entity => entity.Name == "Traveler").Fields.Any(field => field.Name == "notes"))
    throw new InvalidOperationException("Unannotated fields should not be exposed.");


## Configure through the fluent API

Fluent configuration can override attributes or configure otherwise unannotated types. Here it adds `notes`
and changes the description while preserving annotated identity fields, relationships, and ACL mappings.


In [ ]:
var proposedModel = new ContextModelBuilder(annotatedModel.Title, annotatedModel.Description)
    .Entity<TravelerView>(entity => entity
        .Description("Travel customer with application notes")
        .Field(traveler => traveler.Notes, field => field.Name("notes").Text()))
    .Entity<FlightView>(entity => entity.KeyTemplate("flight-reservation:{id}"))
    .Build();
foreach (var entity in proposedModel.Entities)
{
    Console.WriteLine($"{entity.Name}: {entity.RedisKeyTemplate}");
    foreach (var relationship in entity.Relationships)
        Console.WriteLine($"  {relationship.SourceField} -> {relationship.Target} ({relationship.Name})");
    foreach (var mapping in entity.AclFieldMappings ?? [])
        Console.WriteLine($"  ACL {mapping.DocumentField} -> {mapping.AccessTag}");
}


## Review additive and destructive changes

Compare models before applying a surface update. Adding `notes` is additive; changing a Redis key template is
destructive. Comparison classifies contract changes, but does not migrate stored records or prove data compatibility.


In [ ]:
var additiveChanges = ContextDataModelComparer.Compare(annotatedModel, proposedModel);
foreach (var change in additiveChanges) Console.WriteLine($"{change.Kind}: {change.Description}");
if (!additiveChanges.Any(change => change.Kind == ContextModelChangeKind.Additive))
    throw new InvalidOperationException("Expected an additive field change.");
var changedKeys = proposedModel with
{
    Entities = proposedModel.Entities.Select(entity => entity.Name == "Traveler"
        ? entity with { RedisKeyTemplate = "traveler-v2:{id}" } : entity).ToArray(),
};
var destructiveChanges = ContextDataModelComparer.Compare(proposedModel, changedKeys);
foreach (var change in destructiveChanges) Console.WriteLine($"{change.Kind}: {change.Description}");
if (!destructiveChanges.Any(change => change.Kind == ContextModelChangeKind.Destructive))
    throw new InvalidOperationException("Expected a destructive key change.");
var proposedPath = ContextNotebook.ArtifactPath("notebook-projection-review-only.json");
await ContextDataModelJson.WriteAsync(proposedModel, proposedPath);
Console.WriteLine("Saved review-only projection. The deployed surface has not changed.");


## Inspect the complete deployment artifact

The setup provisioner compiles all four sample POCOs and calls `ctxctl surface create/update --datamodel ...`.
The optional `Redis.Iris.ContextRetriever.Management` package exposes `CreateSurfaceAsync` and `UpdateSurfaceAsync`
for direct HTTP administration with an explicit admin API key. Neither compilation nor creation imports POCO instances; the separate seeder does that.


In [ ]:
var fullModel = ContextDataModelJson.Deserialize(await File.ReadAllTextAsync(
    ContextNotebook.ArtifactPath("redis-iris-travel-context-model.json")));
Console.WriteLine($"Full fixture: {fullModel.EntityCount} entities");
foreach (var entity in fullModel.Entities) Console.WriteLine($"- {entity.Name}");


## Read a typed traveler over the authorized session

The local projection determines the entity/get-tool name and deserialization type. The server's deployed model
and scoped key determine what is actually accessible. A local C# projection cannot expand that authorization.


In [ ]:
var traveler = ContextNotebook.LoadTraveler();
var contextClient = await traveler.ConnectAsync();
try
{
    var typed = new ContextRetrieverTypedClient(contextClient, annotatedModel);
    var customer = await typed.GetByIdAsync<TravelerView>(traveler.TravelerId)
        ?? throw new InvalidOperationException("Expected the seeded traveler.");
    if (customer.Id != traveler.TravelerId || customer.TenantId != traveler.TenantId)
        throw new InvalidOperationException("Unexpected traveler returned.");
    Console.WriteLine($"{customer.FullName} ({customer.Id}) in {customer.TenantId}");
}
finally { await contextClient.DisposeAsync(); }
